<small><b>04 Sentiment / Positioning</b> (optional) — Capture product “vibe”: <code>tone</code>, <code>positioning</code>, and upvote-based <code>heat_tier</code> (High Heat vs Long-tail Utility). Writes <code>data/saas_positioned.csv</code>.</small>

<small><b>Step 0 — Setup</b>. Prefer <code>saas_enriched.csv</code> from notebook 03. Zero-shot reuses <code>facebook/bart-large-mnli</code>. Set <code>USE_LLM_POSITIONING=True</code> only if you want DeepSeek overrides.</small>

In [2]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=True)

from src.positioning import (
    POSITIONED_PATH,
    POSITIONING_LABELS,
    TONE_LABELS,
    enrich_positioning,
    filter_positioning,
    load_base,
)
from src.classify import ENRICHED_PATH, CLEANED_PATH

SAMPLE_SIZE = 40
USE_ZERO_SHOT = True
USE_LLM_POSITIONING = False  # True → DeepSeek positioning labels

print("base prefers:", ENRICHED_PATH if ENRICHED_PATH.exists() else CLEANED_PATH)
print("POSITIONED_PATH:", POSITIONED_PATH)
print("TONE_LABELS:", TONE_LABELS)
print("POSITIONING_LABELS:", POSITIONING_LABELS)
print("SAMPLE_SIZE:", SAMPLE_SIZE)
print("DEEPSEEK_API_KEY set:", bool(os.getenv("DEEPSEEK_API_KEY")))

base prefers: C:\Users\suxia\Desktop\Saas-Recommender(2026)\data\saas_enriched.csv
POSITIONED_PATH: C:\Users\suxia\Desktop\Saas-Recommender(2026)\data\saas_positioned.csv
TONE_LABELS: ['joy', 'professional', 'innovative', 'practical']
POSITIONING_LABELS: ['Highly Viral', 'Niche', 'Enterprise-Ready', 'Indie Hacker Friendly']
SAMPLE_SIZE: 40
DEEPSEEK_API_KEY set: True


<small><b>Step 1 — Load</b> enriched (or cleaned) products.</small>

In [3]:
df = load_base(sample_size=SAMPLE_SIZE)
print("rows:", len(df))
display(df[["name", "tagline", "votes_count", "is_viral"]].head(5) if "is_viral" in df.columns else df.head(5))
print("votes_count describe:")
display(df["votes_count"].describe())

rows: 40


,name,tagline,votes_count,is_viral
0,Bluedot 2.1,Record on Apple Watch. Sync with Claude,247,1
1,Powabase,"Build AI apps with Postgres, RAG, and agents",223,1
2,Oasis Browser for Mac,A privacy-first AI browser you can train anony...,173,1
3,zero.xyz,"Give your AI agent access to ~8k tools, APIs a...",160,1
4,Coworker AI,More AI for less spend with context-aware mode...,138,1


votes_count describe:


count     40.000000
mean      78.400000
std       52.263533
min       11.000000
25%       62.750000
50%       72.500000
75%       85.750000
max      247.000000
Name: votes_count, dtype: float64

<small><b>Step 2 — Heat tier from upvotes</b> (no model): <code>High Heat</code> vs <code>Long-tail Utility</code>. Uses votes threshold + <code>is_viral</code>.</small>

In [4]:
from src.positioning import add_heat_labels

heat_df = add_heat_labels(df)
print(heat_df["heat_tier"].value_counts())
display(heat_df[["name", "votes_count", "is_viral", "heat_tier"]].sort_values("votes_count", ascending=False).head(10))

heat_tier
High Heat    40
Name: count, dtype: int64


,name,votes_count,is_viral,heat_tier
0,Bluedot 2.1,247,1,High Heat
1,Powabase,223,1,High Heat
2,Oasis Browser for Mac,173,1,High Heat
3,zero.xyz,160,1,High Heat
4,Coworker AI,138,1,High Heat
5,Octolane,126,1,High Heat
6,Mojito,113,1,High Heat
7,Layers,97,1,High Heat
8,Calling Skills for AI Agents,91,1,High Heat
9,Pawse.ai,88,1,High Heat


<small><b>Step 3 — Tone + positioning</b>. Zero-shot on tagline+description: tone ∈ joy/professional/innovative/practical; positioning ∈ Highly Viral / Niche / Enterprise-Ready / Indie Hacker Friendly. Optional DeepSeek override.</small>

In [5]:
print("Starting positioning enrich...", flush=True)
positioned = enrich_positioning(
    df,
    use_zero_shot=USE_ZERO_SHOT,
    use_llm_positioning=USE_LLM_POSITIONING,
    show_progress=True,
)
cols = [
    "name", "votes_count", "heat_tier",
    "tone", "tone_score",
    "positioning", "positioning_score", "positioning_source",
]
display(positioned[cols].head(10))
print("\ntone:"); display(positioned["tone"].value_counts())
print("positioning:"); display(positioned["positioning"].value_counts())
print("heat_tier:"); display(positioned["heat_tier"].value_counts())

Starting positioning enrich...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

[positioning/zero-shot] 1/40: Bluedot 2.1
[positioning/zero-shot] 10/40: Pawse.ai
[positioning/zero-shot] 20/40: Extend
[positioning/zero-shot] 30/40: AgenticCalling AI
[positioning/zero-shot] 40/40: Syncaut


,name,votes_count,heat_tier,tone,tone_score,positioning,positioning_score,positioning_source
0,Bluedot 2.1,247,High Heat,practical,0.4933,Indie Hacker Friendly,0.3204,zero-shot
1,Powabase,223,High Heat,innovative,0.5817,Niche,0.4054,zero-shot
2,Oasis Browser for Mac,173,High Heat,innovative,0.6203,Niche,0.7303,zero-shot
3,zero.xyz,160,High Heat,practical,0.3939,Niche,0.3834,zero-shot
4,Coworker AI,138,High Heat,innovative,0.5510,Niche,0.3812,zero-shot
5,Octolane,126,High Heat,innovative,0.7855,Indie Hacker Friendly,0.3103,zero-shot
6,Mojito,113,High Heat,practical,0.6628,Indie Hacker Friendly,0.4349,zero-shot
7,Layers,97,High Heat,innovative,0.6650,Indie Hacker Friendly,0.3615,zero-shot
8,Calling Skills for AI Agents,91,High Heat,practical,0.3813,Niche,0.5344,zero-shot
9,Pawse.ai,88,High Heat,innovative,0.4535,Niche,0.3690,zero-shot



tone:


tone
innovative    28
practical     12
Name: count, dtype: int64

positioning:


positioning
Niche                    23
Indie Hacker Friendly    14
Highly Viral              2
Enterprise-Ready          1
Name: count, dtype: int64

heat_tier:


heat_tier
High Heat    40
Name: count, dtype: int64

<small><b>Step 4 — Filter demo</b>. Example: High Heat + innovative, or Long-tail Utility + Indie Hacker Friendly.</small>

In [6]:
hot = filter_positioning(positioned, heat_tier="High Heat", tone="innovative")
print("High Heat + innovative →", len(hot))
display(hot[cols].head(8))

quiet = filter_positioning(positioned, heat_tier="Long-tail Utility", positioning="Indie Hacker Friendly")
print("\nLong-tail Utility + Indie Hacker Friendly →", len(quiet))
display(quiet[cols].head(8))

High Heat + innovative → 28


,name,votes_count,heat_tier,tone,tone_score,positioning,positioning_score,positioning_source
0,Powabase,223,High Heat,innovative,0.5817,Niche,0.4054,zero-shot
1,Oasis Browser for Mac,173,High Heat,innovative,0.6203,Niche,0.7303,zero-shot
2,Coworker AI,138,High Heat,innovative,0.5510,Niche,0.3812,zero-shot
3,Octolane,126,High Heat,innovative,0.7855,Indie Hacker Friendly,0.3103,zero-shot
4,Layers,97,High Heat,innovative,0.6650,Indie Hacker Friendly,0.3615,zero-shot
5,Pawse.ai,88,High Heat,innovative,0.4535,Niche,0.3690,zero-shot
6,CircadiaOS,84,High Heat,innovative,0.5816,Niche,0.4917,zero-shot
7,Krater,83,High Heat,innovative,0.6490,Highly Viral,0.4248,zero-shot



Long-tail Utility + Indie Hacker Friendly → 0


,name,votes_count,heat_tier,tone,tone_score,positioning,positioning_score,positioning_source


<small><b>Step 5 — Export</b> <code>data/saas_positioned.csv</code> for Gradio vibe filters.</small>

In [7]:
POSITIONED_PATH.parent.mkdir(parents=True, exist_ok=True)
positioned.to_csv(POSITIONED_PATH, index=False)
print(f"Wrote {POSITIONED_PATH} -> {positioned.shape[0]} rows, {positioned.shape[1]} cols", flush=True)

Wrote C:\Users\suxia\Desktop\Saas-Recommender(2026)\data\saas_positioned.csv -> 40 rows, 49 cols
